# Strava Performance Dashboard — Exploratory Data Analysis
Understanding the structure, quality, and patterns in 571 runs logged since 2021.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("../data/raw/activities.csv", parse_dates=["date"])
print(f"Loaded {len(df)} runs")
df.head()

Loaded 571 runs


,id,date,name,distance_km,duration_sec,elevation_m,avg_hr,max_hr,avg_speed,pace_min_per_km
0,5159424797,2021-04-20 12:50:09+00:00,Lunch Run,3.0163,925,0.0,NaN,NaN,3.261,5.111118
1,5207562011,2021-04-28 16:18:00+00:00,Afternoon Run,2.0068,594,0.0,NaN,NaN,3.378,4.933227
2,6227335431,2021-11-08 00:05:57+00:00,Midnight run,6.4928,2041,32.2,NaN,NaN,3.181,5.239137
3,6944184769,2022-04-07 12:31:22+00:00,Lunch Run,2.5614,706,0.0,NaN,NaN,3.628,4.593842
4,7022950636,2022-04-22 16:16:56+00:00,Afternoon Run,2.0087,478,0.0,NaN,NaN,4.202,3.966081


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 571 entries, 0 to 570
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   id               571 non-null    int64              
 1   date             571 non-null    datetime64[us, UTC]
 2   name             571 non-null    str                
 3   distance_km      571 non-null    float64            
 4   duration_sec     571 non-null    int64              
 5   elevation_m      571 non-null    float64            
 6   avg_hr           330 non-null    float64            
 7   max_hr           330 non-null    float64            
 8   avg_speed        571 non-null    float64            
 9   pace_min_per_km  571 non-null    float64            
dtypes: datetime64[us, UTC](1), float64(6), int64(2), str(1)
memory usage: 55.6 KB


In [3]:
df.isnull().sum()

id                   0
date                 0
name                 0
distance_km          0
duration_sec         0
elevation_m          0
avg_hr             241
max_hr             241
avg_speed            0
pace_min_per_km      0
dtype: int64

## HR Data Coverage
241 runs (42%) are missing heart rate data — likely pre-HR monitor period.
Let's see when HR tracking started.

In [4]:
hr_start = df[df["avg_hr"].notna()]["date"].min()
print(f"HR tracking started: {hr_start.date()}")
print(f"Runs with HR: {df['avg_hr'].notna().sum()}")
print(f"Runs without HR: {df['avg_hr'].isna().sum()}")

HR tracking started: 2023-03-16
Runs with HR: 330
Runs without HR: 241


In [21]:
df["week"] = df["date"].dt.tz_localize(None).dt.to_period("W").dt.start_time
weekly = df.groupby("week")["distance_km"].sum().reset_index()

fig = px.bar(weekly, x="week", y="distance_km",
             title="Weekly Mileage Over Time",
             labels={"distance_km": "km", "week": "Week"})
fig.show()

In [11]:
fig = px.scatter(df, x="date", y="pace_min_per_km",
                 title="Pace Over Time (min/km)",
                 trendline="lowess",
                 labels={"pace_min_per_km": "Pace (min/km)", "date": "Date"})
fig.update_yaxes(autorange="reversed")  # lower pace = faster
fig.show()

In [9]:
# Filter out walks misclassified as runs
df = df[df["pace_min_per_km"] <= 8].reset_index(drop=True)
print(f"Runs remaining after cleaning: {len(df)}")

Runs remaining after cleaning: 516


In [10]:
df = df[(df["pace_min_per_km"] <= 8) & (df["distance_km"] >= 1)].reset_index(drop=True)
print(f"Runs remaining after cleaning: {len(df)}")

Runs remaining after cleaning: 509


In [12]:
fig = px.histogram(df, x="distance_km", nbins=40,
                   title="Distribution of Run Distances",
                   labels={"distance_km": "Distance (km)"})
fig.show()

In [13]:
df_hr = df[df["avg_hr"].notna()]

fig = px.histogram(df_hr, x="avg_hr", nbins=40,
                   title="Distribution of Average Heart Rate",
                   labels={"avg_hr": "Avg HR (bpm)"})
fig.show()

In [14]:
fig = px.scatter(df, x="distance_km", y="pace_min_per_km",
                 color="elevation_m",
                 title="Pace vs Distance (coloured by elevation)",
                 labels={"distance_km": "Distance (km)",
                         "pace_min_per_km": "Pace (min/km)",
                         "elevation_m": "Elevation (m)"})
fig.update_yaxes(autorange="reversed")
fig.show()

In [15]:
df["day_of_week"] = df["date"].dt.day_name()
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

day_counts = df["day_of_week"].value_counts().reindex(day_order)
fig = px.bar(day_counts, 
             title="Runs by Day of Week",
             labels={"value": "Number of runs", "index": "Day"})
fig.show()

In [16]:
df["month"] = df["date"].dt.month_name()
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]

month_counts = df["month"].value_counts().reindex(month_order)
fig = px.bar(month_counts,
             title="Runs by Month",
             labels={"value": "Number of runs", "index": "Month"})
fig.show()

In [17]:
# Remove physiologically impossible HR values
df.loc[df["avg_hr"] < 95, "avg_hr"] = None
df.loc[df["max_hr"] < 95, "max_hr"] = None

print(f"Clean HR runs: {df['avg_hr'].notna().sum()}")

Clean HR runs: 310


In [20]:
df.to_csv("../data/processed/activities_clean.csv", index=False)
print("Saved cleaned data")

Saved cleaned data


In [6]:
import folium
from folium.plugins import HeatMap
import pandas as pd

# Load GPS data
gps = pd.read_csv("../data/raw/gps_streams.csv")
print(f"GPS points: {len(gps)}")

# Drop nulls
gps = gps.dropna(subset=["lat", "lng"])

# Centre map on your most common location
centre_lat = gps["lat"].median()
centre_lng = gps["lng"].median()
print(f"Map centre: {centre_lat:.4f}, {centre_lng:.4f}")

# Build heatmap
m = folium.Map(
    location=[centre_lat, centre_lng],
    zoom_start=13,
    tiles="CartoDB dark_matter"  # dark background looks great for heatmaps
)

# Prepare points
heat_data = gps[["lat", "lng"]].values.tolist()

HeatMap(
    heat_data,
    radius=8,
    blur=10,
    max_zoom=13,
    min_opacity=0.3
).add_to(m)

# Save
m.save("../data/processed/heatmap.html")

GPS points: 330572
Map centre: 50.8680, 4.6992
